# Model: mistralai/Mistral-7B-Instruct-v0.3

**RMIT Hackathon 2026 – Challenge 1** – QLoRA 4-bit + classification head.

Pipeline tự động: kiểm tra input -> smoke test suy luận -> đo tốc độ -> tự tính số step theo `TRAIN_BUDGET_MIN` -> train -> suy luận -> `submission.csv`.

In [ ]:
# Model: mistralai/Mistral-7B-Instruct-v0.3
import os, sys, subprocess
os.environ["CUDA_VISIBLE_DEVICES"] = "0"       # phải đặt TRƯỚC khi import torch (1 GPU, tránh DataParallel với 4-bit)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

def ensure(pkg):                                # Internet đang bật -> tự cài nếu môi trường thiếu
    try: __import__(pkg)
    except ImportError:
        print("Cài", pkg); subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
for _p in ["accelerate", "peft", "bitsandbytes"]: ensure(_p)

import glob, time, random, math
import numpy as np, pandas as pd, torch
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
import transformers, peft, bitsandbytes
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, BitsAndBytesConfig,
                          TrainingArguments, Trainer, TrainerCallback)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
print("transformers", transformers.__version__, "| peft", peft.__version__,
      "| bitsandbytes", bitsandbytes.__version__, "| torch", torch.__version__)

# ================== CẤU HÌNH ==================
DEBUG        = False       # True chỉ để thử nhanh trên vài trăm dòng
DEBUG_TRAIN, DEBUG_VAL = 320, 100

MODEL_KEY    = "mistral-7b-instruct-v0.3"
EXPLICIT     = f"/kaggle/input/models/aisuko/{MODEL_KEY}/transformers/default/1"
TAG          = "mistral7b"

# >>> CHỈNH Ô NÀY THEO THỜI GIAN CÒN LẠI <<<
# Tổng thời gian ~ 10 (load model) + TRAIN_BUDGET_MIN + ~0.3*TRAIN_BUDGET_MIN (suy luận) + 10 (nộp).
# => TRAIN_BUDGET_MIN <= (số phút còn lại - 30) / 1.3.   Ví dụ còn 4 giờ (240 phút) -> ~160.
TRAIN_BUDGET_MIN = 300

TRAIN_ROWS   = None        # None = dùng hết; thường không cần chỉnh vì max_steps tự tính theo ngân sách
MAXLEN, HEAD_TOK = 256, 192
EPOCHS       = 1
LR           = 1e-4
BS, GRAD_ACC = 4, 4        # batch hiệu dụng 16; nếu OOM -> BS=2, GRAD_ACC=8
LORA_R, LORA_ALPHA = 16, 32
VAL_SIZE     = 0.10
SEED         = 42
WORK         = "/kaggle/working"
RESUME_FROM  = None
# ==============================================

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all(SEED)
print("GPU:", torch.cuda.get_device_name(0), "| VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory/2**30, 1))

In [ ]:
def find_data():
    p = "/kaggle/input/competitions/rmit-hackathon-2026"
    if os.path.exists(f"{p}/train.csv"): return p
    hits = glob.glob("/kaggle/input/**/train.csv", recursive=True)
    assert hits, "Không thấy train.csv -> Add Input cuộc thi"
    return os.path.dirname(hits[0])

def find_model(key, explicit=None, fallback=None):
    import re
    if explicit and os.path.exists(f"{explicit}/config.json"): return explicit
    norm = lambda s: re.sub(r"[^a-z0-9]", "", s.lower())
    cfgs = glob.glob("/kaggle/input/**/config.json", recursive=True)
    for k in [key] + ([fallback] if fallback else []):
        hits = [os.path.dirname(c) for c in cfgs if norm(k) in norm(c)]
        if hits: return sorted(hits)[0]
    raise AssertionError(
        f"Không thấy model '{key}' trong /kaggle/input. Add Input > Models > tìm '{key}' (owner aisuko). "
        f"Các thư mục có config.json hiện có: {[os.path.dirname(c) for c in cfgs]}")

DATA   = find_data()
MODEL = find_model(MODEL_KEY, EXPLICIT, fallback='mistral')   # fail-fast
LABELS = ["BENIGN", "JAILBREAK", "PROMPT_INJECTION"]
L2I    = {l: i for i, l in enumerate(LABELS)}

train = pd.read_csv(f"{DATA}/train.csv")
test  = pd.read_csv(f"{DATA}/test.csv")
print(train.shape, test.shape)
print(train.label.value_counts(normalize=True).round(3).to_dict())
print(pd.crosstab(train.language, train.label))
print(train.text.str.len().describe().round(0).to_dict())

# Chia validation: GIỮ NGUYÊN seed/tham số ở mọi notebook để ensemble dùng cùng tập valid
strat = train.label + "_" + train.language
tr, va = train_test_split(train, test_size=VAL_SIZE, stratify=strat, random_state=42)
tr = tr.reset_index(drop=True); va = va.reset_index(drop=True)
if DEBUG:
    tr = tr.sample(DEBUG_TRAIN, random_state=0).reset_index(drop=True)
    va = va.sample(DEBUG_VAL,   random_state=0).reset_index(drop=True)
test_run = test.head(100).copy() if DEBUG else test
ytr = tr.label.map(L2I).values
yva = va.label.map(L2I).values
print("train/valid/test:", len(tr), len(va), len(test_run))

# Class weight: căn bậc hai của "balanced" (đủ mạnh cho lớp 7%, không quá tay); sẽ chỉnh thêm bằng bias ở bước sau
cw = compute_class_weight("balanced", classes=np.arange(3), y=ytr) ** 0.5
CW = torch.tensor(cw, dtype=torch.float32)
print("class weights:", cw.round(3))

if (not DEBUG) and TRAIN_ROWS and TRAIN_ROWS < len(tr):
    tr, _ = train_test_split(tr, train_size=TRAIN_ROWS, stratify=tr.label + "_" + tr.language, random_state=0)
    tr = tr.reset_index(drop=True); ytr = tr.label.map(L2I).values
    print("Subsample train ->", len(tr))

In [ ]:
# ---------- Tokenizer + prompt: [lang] text (head/tail) + hậu tố câu hỏi ----------
# MODEL đã được tìm ở ô trước (fail-fast)
print("MODEL:", MODEL)
tok = AutoTokenizer.from_pretrained(MODEL)
tok.model_max_length = 10**6
if tok.pad_token is None: tok.pad_token = tok.eos_token
tok.padding_side = "right"
PAD, BOS = tok.pad_token_id, tok.bos_token_id

SUFFIX = tok("\n\nClassify (BENIGN / JAILBREAK / PROMPT_INJECTION):", add_special_tokens=False)["input_ids"]
KEEP = MAXLEN - 1 - len(SUFFIX)

def encode_df(df):
    texts = ("[" + df.language + "] " + df.text).tolist()
    ids = tok(texts, add_special_tokens=False, truncation=False)["input_ids"]
    out, ncut = [], 0
    for x in ids:
        if len(x) > KEEP:
            x = x[:HEAD_TOK] + x[-(KEEP - HEAD_TOK):]; ncut += 1
        out.append([BOS] + x + SUFFIX)    # token cuối luôn là hậu tố -> hidden state cuối "biết" nhiệm vụ
    print(f"  cắt head+tail: {ncut}/{len(ids)} dòng")
    return out

ids_tr, ids_va, ids_te = encode_df(tr), encode_df(va), encode_df(test_run)

def arrange_by_length(lengths, bs, chunk=64, seed=0):
    """Gom các dòng cùng độ dài vào cùng batch (giảm padding ~2x), đặt batch dài nhất lên đầu để OOM lộ sớm."""
    rng = np.random.RandomState(seed); idx = rng.permutation(len(lengths)); batches = []
    for i in range(0, len(idx), bs * chunk):
        c = idx[i:i + bs * chunk]; c = c[np.argsort([-lengths[j] for j in c])]
        batches += [c[k:k + bs] for k in range(0, len(c), bs)]
    random.Random(seed).shuffle(batches)
    j = int(np.argmax([max(lengths[k] for k in b) for b in batches]))
    batches.insert(0, batches.pop(j))
    return np.concatenate(batches)

order = arrange_by_length([len(x) for x in ids_tr], BS)
ids_tr_arr = [ids_tr[i] for i in order]; ytr_arr = ytr[order]

class DS(torch.utils.data.Dataset):
    def __init__(self, ids, labels): self.ids, self.labels = ids, labels
    def __len__(self): return len(self.ids)
    def __getitem__(self, i): return {"input_ids": self.ids[i], "labels": int(self.labels[i])}

def collate(batch):
    mx = max(len(b["input_ids"]) for b in batch)
    inp = torch.full((len(batch), mx), PAD, dtype=torch.long); att = torch.zeros_like(inp)
    for r, b in enumerate(batch):
        L = len(b["input_ids"]); inp[r, :L] = torch.tensor(b["input_ids"]); att[r, :L] = 1
    return {"input_ids": inp, "attention_mask": att, "labels": torch.tensor([b["labels"] for b in batch])}

ds_tr = DS(ids_tr_arr, ytr_arr)
print("độ dài token: median", int(np.median([len(x) for x in ids_tr])), "| max", max(len(x) for x in ids_tr))

In [ ]:
# ---------- Load Mistral 4-bit + LoRA ----------
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)  # T4 không có bf16
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL, num_labels=3, quantization_config=bnb, device_map={"": 0},
    id2label={i: l for l, i in L2I.items()}, label2id=L2I)
model.config.pad_token_id = PAD
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
lora = LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.05, bias="none", task_type="SEQ_CLS",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
model = get_peft_model(model, lora)      # head "score" tự được train đầy đủ (SEQ_CLS)
model.print_trainable_parameters()
print("VRAM đã dùng (GB):", round(torch.cuda.memory_allocated()/2**30, 2))

# ---------- Hàm suy luận (sắp theo độ dài, lưu logits) ----------
@torch.no_grad()
def predict_logits(ids_list, bs=16):
    model.eval(); model.config.use_cache = False
    order = np.argsort([len(x) for x in ids_list]); out = np.zeros((len(ids_list), 3), dtype=np.float32)
    for i in range(0, len(order), bs):
        b = order[i:i + bs]; mx = max(len(ids_list[j]) for j in b)
        inp = torch.full((len(b), mx), PAD, dtype=torch.long); att = torch.zeros_like(inp)
        for r, j in enumerate(b):
            L = len(ids_list[j]); inp[r, :L] = torch.tensor(ids_list[j]); att[r, :L] = 1
        with torch.autocast("cuda", dtype=torch.float16):
            lg = model(input_ids=inp.cuda(), attention_mask=att.cuda()).logits
        out[b] = lg.float().cpu().numpy()
    return out

# Smoke test: bắt lỗi suy luận NGAY BÂY GIỜ, không phải sau nhiều giờ train
_t = time.time(); _x = predict_logits(ids_va[:32])
assert _x.shape == (32, 3) and np.isfinite(_x).all(), "Suy luận lỗi / ra NaN"
print(f"Smoke test suy luận OK ({(time.time()-_t)/32*1000:.0f} ms/mẫu, chưa train)")

In [ ]:
# ---------- Hiệu chuẩn tốc độ -> tự tính max_steps ----------
class WTrainerBase(Trainer):
    def _get_train_sampler(self, *a, **k):          # giữ thứ tự đã gom theo độ dài
        return torch.utils.data.SequentialSampler(self.train_dataset)
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs["labels"]
        out = model(**{k: v for k, v in inputs.items() if k != "labels"})
        loss = F.cross_entropy(out.logits.float(), labels, weight=CW.to(out.logits.device))
        return (loss, out) if return_outputs else loss

def calibrate(n=12):
    """Chạy forward+backward (KHÔNG cập nhật trọng số) trên batch dài nhất + n batch ngẫu nhiên."""
    nb = len(ds_tr) // BS
    g = torch.Generator().manual_seed(0)
    pick = [0] + (torch.randperm(nb - 1, generator=g)[:min(n, nb - 1)] + 1).tolist()
    model.train(); torch.cuda.reset_peak_memory_stats(); times = []
    for k, bi in enumerate(pick):
        b = collate([ds_tr[j] for j in range(bi * BS, bi * BS + BS)])
        b = {a: v.cuda() for a, v in b.items()}
        torch.cuda.synchronize(); t = time.time()
        with torch.autocast("cuda", dtype=torch.float16):
            lg = model(input_ids=b["input_ids"], attention_mask=b["attention_mask"]).logits
        F.cross_entropy(lg.float(), b["labels"], weight=CW.cuda()).backward()
        torch.cuda.synchronize()
        if k > 0: times.append(time.time() - t)      # batch đầu = dài nhất + warm-up, chỉ dùng để thử OOM
        model.zero_grad(set_to_none=True)
    print(f"Calibrate: {np.mean(times):.2f}s/micro-batch | VRAM peak {torch.cuda.max_memory_allocated()/2**30:.1f}GB (batch dài nhất đã qua)")
    return float(np.mean(times))

eff_bs = BS * GRAD_ACC
mb_sec = calibrate()
sec_per_step = mb_sec * GRAD_ACC * 1.15              # +15% cho optimizer/overhead
steps_full = math.ceil(len(ds_tr) / eff_bs) * EPOCHS
steps_fit  = max(int(TRAIN_BUDGET_MIN * 60 / sec_per_step), 10)
MAX_STEPS  = min(steps_full, steps_fit)
print(f"~{sec_per_step:.1f}s/step | full={steps_full} step (~{steps_full*sec_per_step/3600:.1f}h) | "
      f"ngân sách {TRAIN_BUDGET_MIN} phút -> chạy {MAX_STEPS} step (~{MAX_STEPS*eff_bs} mẫu, "
      f"{100*MAX_STEPS/steps_full:.0f}% 1 epoch)")
if MAX_STEPS < 0.3 * steps_full:
    print("CẢNH BÁO: chỉ train được <30% dữ liệu. Cân nhắc tăng TRAIN_BUDGET_MIN hoặc giảm MAXLEN xuống 192.")

# ---------- Trainer ----------
class Progress(TrainerCallback):
    def on_train_begin(self, args, state, control, **kw): self.t0 = time.time()
    def on_step_end(self, args, state, control, **kw):
        el = time.time() - self.t0
        if state.global_step % 10 == 0:
            sps = el / max(state.global_step, 1)
            print(f"step {state.global_step}/{state.max_steps} | {sps:.1f}s/step | ETA {(state.max_steps-state.global_step)*sps/60:.0f} phút", flush=True)
        if el > TRAIN_BUDGET_MIN * 60 * 1.3:          # chốt chặn an toàn
            print("Quá 130% ngân sách -> dừng và lưu."); control.should_training_stop = True

args = TrainingArguments(
    output_dir=f"{WORK}/ckpt_{TAG}", num_train_epochs=EPOCHS, max_steps=MAX_STEPS,
    per_device_train_batch_size=BS, gradient_accumulation_steps=GRAD_ACC,
    learning_rate=LR, lr_scheduler_type="cosine", warmup_steps=max(int(0.05 * MAX_STEPS), 1),
    weight_decay=0.0, max_grad_norm=1.0, optim="paged_adamw_8bit",
    fp16=True, gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
    save_strategy="steps", save_steps=100, save_total_limit=2,
    logging_steps=10, report_to="none", seed=SEED,
    dataloader_num_workers=2, remove_unused_columns=False, label_names=["labels"],
)
trainer = WTrainerBase(model=model, args=args, train_dataset=ds_tr, data_collator=collate, callbacks=[Progress()])
t0 = time.time(); trainer.train(resume_from_checkpoint=RESUME_FROM)
print(f"Train xong: {(time.time()-t0)/60:.1f} phút")
model.save_pretrained(f"{WORK}/mistral_lora")

In [ ]:
# ---------- Suy luận valid + test ----------
t0 = time.time()
valid_logits = predict_logits(ids_va); print(f"valid xong: {(time.time()-t0)/60:.1f} phút")
np.save(f"{WORK}/{TAG}_valid_logits_raw.npy", valid_logits)     # lưu sớm phòng lỗi ở bước sau
test_logits  = predict_logits(ids_te); print(f"test xong:  {(time.time()-t0)/60:.1f} phút (tích lũy)")
np.save(f"{WORK}/{TAG}_test_logits_raw.npy", test_logits)

In [ ]:
def softmax_np(x):
    e = np.exp(x - x.max(1, keepdims=True)); return e / e.sum(1, keepdims=True)

def report(y, pred, df, name):
    print(f"\n===== {name} | macro F1 = {f1_score(y, pred, average='macro'):.4f} =====")
    print(classification_report(y, pred, target_names=LABELS, digits=4))
    for lang, g in df.groupby("language"):
        ix = g.index.values
        print(f"  {lang}: n={len(ix):4d}  macroF1={f1_score(y[ix], pred[ix], average='macro'):.4f}")

def tune_bias(logits, y, grid=np.arange(-1.5, 1.51, 0.25)):
    """Cộng bias vào log-prob của JAILBREAK / PROMPT_INJECTION để tối đa macro F1 trên valid."""
    lp = np.log(softmax_np(logits) + 1e-9)
    base = f1_score(y, lp.argmax(1), average="macro")
    best, bb = base, (0.0, 0.0)
    for b1 in grid:
        for b2 in grid:
            s = f1_score(y, (lp + np.array([0, b1, b2])).argmax(1), average="macro")
            if s > best + 1e-4: best, bb = s, (b1, b2)
    return base, best, np.array([0, *bb])

pv_raw = valid_logits.argmax(1)
report(yva, pv_raw, va, f"{TAG} - valid (argmax)")

base, best, bias = tune_bias(valid_logits, yva)
print(f"\nBias tuning: {base:.4f} -> {best:.4f}  bias={bias}")
# Chỉ dùng bias nếu lợi rõ rệt (tránh overfit valid)
USE_BIAS = (best - base) >= 0.003
print("Áp dụng bias:", USE_BIAS)
bias_final = bias if USE_BIAS else np.zeros(3)

np.save(f"{WORK}/{TAG}_valid_logits.npy", valid_logits)
np.save(f"{WORK}/{TAG}_test_logits.npy",  test_logits)
va[["id"]].to_csv(f"{WORK}/{TAG}_valid_ids.csv", index=False)   # để ensemble khớp thứ tự
test_run[["id"]].to_csv(f"{WORK}/{TAG}_test_ids.csv", index=False)

lp_te = np.log(softmax_np(test_logits) + 1e-9) + bias_final
pred_te = lp_te.argmax(1)
pred_raw_te = test_logits.argmax(1)

if DEBUG:
    print("DEBUG: bỏ qua ghi submission. Phân bố:", pd.Series(pred_te).map(dict(enumerate(LABELS))).value_counts(normalize=True).round(3).to_dict())
else:
    sub = pd.read_csv(f"{DATA}/sample_submission.csv")
    for fn, p in [("submission.csv", pred_te), ("submission_raw.csv", pred_raw_te)]:
        m = dict(zip(test.id, [LABELS[i] for i in p]))
        s = sub.copy(); s["label"] = s["id"].map(m)
        assert len(s) == 6000 and s["label"].isin(LABELS).all() and s["id"].is_unique
        s.to_csv(f"{WORK}/{fn}", index=False)
        print(fn, s.label.value_counts(normalize=True).round(3).to_dict())
    print("Mục tiêu tham khảo ~ BENIGN 0.62 / JAILBREAK 0.31 / PROMPT_INJECTION 0.07")
    print("-> Nộp file submission.csv (đã áp bias nếu có lợi).")
# Dọn checkpoint trung gian (giữ lại adapter LoRA + logits) cho output gọn
import shutil
shutil.rmtree(f"{WORK}/ckpt_{TAG}", ignore_errors=True)
print("Files output:", sorted(os.listdir(WORK)))
